In [ ]:
!pip install -q vllm

In [ ]:
!rm -rf /kaggle/working/nemotron-reasoning
!git clone https://github.com/josaiahsyiem/nemotron-reasoning.git

import sys
sys.path.insert(0, "/kaggle/working/nemotron-reasoning")

from vcd.solvers.registry import get_solver, all_types
from vcd.verify.extract import extract_boxed
from vcd.detect import detect_type
print("Solvers:", all_types())

In [ ]:
import pandas as pd

CSV_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(CSV_PATH)
df["type"] = df["prompt"].apply(detect_type)
print("Total puzzles:", len(df))

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-7B-Instruct",
    dtype="float16",             # T4 doesn't support bfloat16, use float16
    tensor_parallel_size=2,      # use both T4 GPUs
    gpu_memory_utilization=0.90,
    max_model_len=8192,
)
print("vLLM model loaded!")

In [ ]:
INSTRUCTION = (
    "\n\nSolve this step by step. "
    "Put your final answer inside \\boxed{}. "
    "For example: \\boxed{your answer}"
)

sampling = SamplingParams(temperature=0.7, max_tokens=1024)

# build the chat-formatted prompt for one puzzle
tok = llm.get_tokenizer()
row = df.iloc[3]   # a numeral puzzle
messages = [{"role": "user", "content": row["prompt"] + INSTRUCTION}]
text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

out = llm.generate([text], sampling)
response = out[0].outputs[0].text
print("Response:\n", response)
print("\nExtracted:", extract_boxed(response))
print("Correct answer:", row["answer"])

In [ ]:
import json, os, time

OUTPUT_PATH = "/kaggle/working/traces.jsonl"
tok = llm.get_tokenizer()

def load_done_ids(path):
    done = set()
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    done.add(json.loads(line)["id"])
                except Exception:
                    pass
    return done

def build_prompt(puzzle_prompt):
    messages = [{"role": "user", "content": puzzle_prompt + INSTRUCTION}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def run_batch_vllm(df, limit=None, batch_size=200):
    """Generate for many puzzles at once with vLLM, verify, save as we go."""
    done = load_done_ids(OUTPUT_PATH)
    todo = df[~df["id"].isin(done)]
    if limit:
        todo = todo.head(limit)
    print(f"Already done: {len(done)} | Processing this run: {len(todo)}")

    sampling = SamplingParams(temperature=0.7, max_tokens=1024)
    start = time.time()
    n_correct = 0

    # process in chunks so we save periodically
    for chunk_start in range(0, len(todo), batch_size):
        chunk = todo.iloc[chunk_start:chunk_start + batch_size]
        prompts = [build_prompt(p) for p in chunk["prompt"]]

        outputs = llm.generate(prompts, sampling)   # ALL at once — this is the speed

        with open(OUTPUT_PATH, "a") as f:
            for (_, row), out in zip(chunk.iterrows(), outputs):
                response = out.outputs[0].text
                predicted = extract_boxed(response)
                solver = get_solver(row["type"])
                correct = solver.verify(predicted, row["answer"])
                if correct:
                    n_correct += 1
                f.write(json.dumps({
                    "id": row["id"], "type": row["type"], "answer": row["answer"],
                    "predicted": predicted, "correct": correct, "response": response,
                }) + "\n")

        elapsed = time.time() - start
        n_done = chunk_start + len(chunk)
        print(f"  {n_done}/{len(todo)} | {n_correct} correct | {n_done/elapsed:.2f}/sec")

    print(f"\nDone. Correct this run: {n_correct}/{len(todo)}")

In [ ]:
print("run_batch_vllm defined?", "run_batch_vllm" in dir())
print("llm exists?", "llm" in dir())
print("INSTRUCTION defined?", "INSTRUCTION" in dir())
print("df rows:", len(df) if "df" in dir() else "NO DF")

# is the output file already full from earlier tests?
import os
if os.path.exists("/kaggle/working/traces.jsonl"):
    n = sum(1 for _ in open("/kaggle/working/traces.jsonl"))
    print("existing traces in file:", n)
else:
    print("no output file yet")

In [ ]:
import os
if os.path.exists("/kaggle/working/traces.jsonl"):
    os.remove("/kaggle/working/traces.jsonl")
    print("Cleared old traces file — starting fresh")
else:
    print("No file to clear")

In [ ]:
run_batch_vllm(df, limit=100, batch_size=100)

In [ ]:
import json
results = [json.loads(l) for l in open("/kaggle/working/traces.jsonl")]
rdf = pd.DataFrame(results)
print("Overall:", f"{rdf['correct'].mean():.1%}")
print("\nBy type:")
print(rdf.groupby("type")["correct"].agg(["mean", "count"]).sort_values("mean"))

In [ ]:
# Run the full dataset but ONLY the easy types (they already pass well)
easy_types = ["numeral_conversion", "gravitational_constant", "unit_conversion"]
easy_df = df[df["type"].isin(easy_types)]
print("Easy-type puzzles to process:", len(easy_df))

run_batch_vllm(easy_df, batch_size=200)

In [ ]:
import json
results = [json.loads(l) for l in open("/kaggle/working/traces.jsonl")]
rdf = pd.DataFrame(results)

# how many correct traces per type do we have now?
correct_only = rdf[rdf["correct"]]
print("Total correct traces:", len(correct_only))
print("\nCorrect traces by type:")
print(correct_only.groupby("type").size())

In [ ]:
# Confirm the file and its size before downloading
import os
path = "/kaggle/working/traces.jsonl"
print("File exists:", os.path.exists(path))
print("Size:", round(os.path.getsize(path)/1024/1024, 2), "MB")
print("Lines:", sum(1 for _ in open(path)))

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv")
from vcd.detect import detect_type
df["type"] = df["prompt"].apply(detect_type)

# show 3 full text-encryption puzzles
enc = df[df["type"] == "text_encryption"].head(3)
for _, row in enc.iterrows():
    print(row["prompt"])
    print("ANSWER:", row["answer"])
    print("=" * 70)

In [ ]:
import shutil, sys
shutil.rmtree("/kaggle/working/nemotron-reasoning", ignore_errors=True)
!git clone https://github.com/josaiahsyiem/nemotron-reasoning.git

# force-reload the modules so the new code takes effect
for mod in list(sys.modules):
    if mod.startswith("vcd"):
        del sys.modules[mod]

sys.path.insert(0, "/kaggle/working/nemotron-reasoning")
from vcd.solvers.registry import get_solver
from vcd.verify.extract import extract_boxed
from vcd.detect import detect_type

# confirm the hint method is there now
s = get_solver("text_encryption")
print("Has augment_prompt:", hasattr(s, "augment_prompt"))

In [ ]:
def build_prompt_with_hint(row):
    solver = get_solver(row["type"])
    # apply the solver's hint (augment_prompt) to the raw puzzle
    augmented = solver.augment_prompt(row["prompt"])
    messages = [{"role": "user", "content": augmented + INSTRUCTION}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def test_type_with_hint(df, ptype, n=30):
    """Run n puzzles of one type WITH hints, report pass rate."""
    sub = df[df["type"] == ptype].head(n)
    prompts = [build_prompt_with_hint(row) for _, row in sub.iterrows()]

    sampling = SamplingParams(temperature=0.7, max_tokens=1024)
    outputs = llm.generate(prompts, sampling)

    n_correct = 0
    for (_, row), out in zip(sub.iterrows(), outputs):
        predicted = extract_boxed(out.outputs[0].text)
        if get_solver(ptype).verify(predicted, row["answer"]):
            n_correct += 1
    print(f"{ptype} WITH hint: {n_correct}/{len(sub)} = {n_correct/len(sub):.0%}")
    return n_correct / len(sub)

In [ ]:
test_type_with_hint(df, "text_encryption", n=30)

In [ ]:
# Look at what's happening on text-encryption puzzles with the hint
sub = df[df["type"] == "text_encryption"].head(30)
prompts = [build_prompt_with_hint(row) for _, row in sub.iterrows()]
sampling = SamplingParams(temperature=0.7, max_tokens=1024)
outputs = llm.generate(prompts, sampling)

# show the first 5 failures in detail
shown = 0
for (_, row), out in zip(sub.iterrows(), outputs):
    response = out.outputs[0].text
    predicted = extract_boxed(response)
    correct = get_solver("text_encryption").verify(predicted, row["answer"])
    if not correct and shown < 5:
        shown += 1
        print(f"--- FAILURE {shown} ---")
        print("Correct answer:", row["answer"])
        print("Model predicted:", predicted)
        # show just the hint part of what we sent
        aug = get_solver("text_encryption").augment_prompt(row["prompt"])
        hint_part = aug.split("HINT:")[1][:300] if "HINT:" in aug else "NO HINT"
        print("Hint we gave:", hint_part)
        print()

In [ ]:
import shutil, sys
shutil.rmtree("/kaggle/working/nemotron-reasoning", ignore_errors=True)
!git clone https://github.com/josaiahsyiem/nemotron-reasoning.git

# force-reload the vcd modules so the new wording takes effect
for mod in list(sys.modules):
    if mod.startswith("vcd"):
        del sys.modules[mod]
sys.path.insert(0, "/kaggle/working/nemotron-reasoning")
from vcd.solvers.registry import get_solver
from vcd.verify.extract import extract_boxed
from vcd.detect import detect_type

# confirm new wording is loaded
s = get_solver("text_encryption")
sample = s.augment_prompt("""x -> a
Now, decrypt the following text: x""")
print("New wording present:", "must NOT be changed" in sample)

In [ ]:
test_type_with_hint(df, "text_encryption", n=30)

In [ ]:
sub = df[df["type"] == "text_encryption"].head(30)
prompts = [build_prompt_with_hint(row) for _, row in sub.iterrows()]
sampling = SamplingParams(temperature=0.7, max_tokens=1024)
outputs = llm.generate(prompts, sampling)

shown = 0
for (_, row), out in zip(sub.iterrows(), outputs):
    predicted = extract_boxed(out.outputs[0].text)
    correct = get_solver("text_encryption").verify(predicted, row["answer"])
    if not correct and shown < 6:
        shown += 1
        aug = get_solver("text_encryption").augment_prompt(row["prompt"])
        # pull the "decoded text is" part
        partial = aug.split("decoded text is: '")[1].split("'")[0] if "decoded text is: '" in aug else "?"
        print(f"FAIL {shown}: answer='{row['answer']}' | partial_decode='{partial}' | predicted='{predicted}'")

In [ ]:
import shutil, sys
shutil.rmtree("/kaggle/working/nemotron-reasoning", ignore_errors=True)
!git clone https://github.com/josaiahsyiem/nemotron-reasoning.git

for mod in list(sys.modules):
    if mod.startswith("vcd"):
        del sys.modules[mod]
sys.path.insert(0, "/kaggle/working/nemotron-reasoning")

from vcd.solvers.registry import get_solver
from vcd.solvers.text_encryption import set_vocab
from vcd.vocab import harvest_vocab
from vcd.verify.extract import extract_boxed

# load vocab from the competition train.csv
CSV_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
vocab = harvest_vocab(CSV_PATH)
set_vocab(vocab)
print("Vocab loaded into solver:", len(vocab), "words")

In [ ]:
def build_encryption_prompt(row):
    """Give the model the Python-cracked partial solution as a strong hint."""
    solver = get_solver("text_encryption")
    decoded, key, steps = solver.crack(row["prompt"])
    key_str = ", ".join(f"{c}->{p}" for c, p in sorted(key.items()))
    step_str = "; ".join(steps)

    hint = (
        f"\n\nThe cipher key is: {key_str}. "
        f"Decoding gives: {step_str}. "
        f"The partial answer is '{decoded}'. "
        f"For any word still containing '?', choose the listed candidate that "
        f"fits the sentence. Output the complete decoded text in \\boxed{{}}."
    )
    messages = [{"role": "user", "content": row["prompt"] + hint}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def test_encryption(df, n=30):
    sub = df[df["type"] == "text_encryption"].head(n)
    prompts = [build_encryption_prompt(row) for _, row in sub.iterrows()]
    sampling = SamplingParams(temperature=0.3, max_tokens=1024)   # lower temp for precision
    outputs = llm.generate(prompts, sampling)
    n_correct = 0
    for (_, row), out in zip(sub.iterrows(), outputs):
        predicted = extract_boxed(out.outputs[0].text)
        if get_solver("text_encryption").verify(predicted, row["answer"]):
            n_correct += 1
    print(f"text_encryption: {n_correct}/{len(sub)} = {n_correct/len(sub):.0%}")
    return n_correct / len(sub)

In [ ]:
test_encryption(df, n=30)

In [ ]:
# show 2 full bit-manipulation puzzles with all their examples
bits = df[df["type"] == "bit_manipulation"].head(2)
for _, row in bits.iterrows():
    print(row["prompt"])
    print("ANSWER:", row["answer"])
    print("=" * 70)

In [ ]:
import shutil, sys
shutil.rmtree("/kaggle/working/nemotron-reasoning", ignore_errors=True)
!git clone https://github.com/josaiahsyiem/nemotron-reasoning.git
for mod in list(sys.modules):
    if mod.startswith("vcd"):
        del sys.modules[mod]
sys.path.insert(0, "/kaggle/working/nemotron-reasoning")
from vcd.solvers.bit_manipulation import BitManipulationSolver

s = BitManipulationSolver()
bits = df[df["type"] == "bit_manipulation"]

fully_solved = 0
correct = 0
partial = 0
for _, row in bits.head(300).iterrows():
    answer, funcs = s.solve(row["prompt"])
    n_bits = sum(1 for n, _ in funcs if n is not None)
    if answer is not None:
        fully_solved += 1
        if s.verify(answer, row["answer"]):
            correct += 1
    else:
        partial += 1

print(f"Out of 300 bit puzzles:")
print(f"  Fully solved by Python: {fully_solved}")
print(f"  Of those, CORRECT: {correct}")
print(f"  Partial (some bits unsolved): {partial}")
print(f"\nPython-alone accuracy: {correct}/300 = {correct/300:.0%}")

In [ ]:
def build_bit_prompt(row):
    solver = get_solver("bit_manipulation")
    trace, answer = solver.generate_trace(row["prompt"])
    hint = (
        f"\n\nAnalysis of the bit pattern: {trace} "
        f"Give the final 8-bit output in \\boxed{{}}."
    )
    messages = [{"role": "user", "content": row["prompt"] + hint}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def test_bits_combined(df, n=100):
    from vcd.solvers.bit_manipulation import BitManipulationSolver
    s = get_solver("bit_manipulation")
    sub = df[df["type"] == "bit_manipulation"].head(n)

    # Python-alone: where it fully solves AND is correct
    py_correct = 0
    for _, row in sub.iterrows():
        ans, _ = s.solve(row["prompt"])
        if ans and s.verify(ans, row["answer"]):
            py_correct += 1

    # Model-with-hint
    prompts = [build_bit_prompt(row) for _, row in sub.iterrows()]
    sampling = SamplingParams(temperature=0.3, max_tokens=1024)
    outputs = llm.generate(prompts, sampling)
    model_correct = 0
    for (_, row), out in zip(sub.iterrows(), outputs):
        pred = extract_boxed(out.outputs[0].text)
        if s.verify(pred, row["answer"]):
            model_correct += 1

    print(f"Python-alone:      {py_correct}/{n} = {py_correct/n:.0%}")
    print(f"Model+Python hint: {model_correct}/{n} = {model_correct/n:.0%}")

test_bits_combined(df, n=100)

In [ ]:
import shutil, sys
shutil.rmtree("/kaggle/working/nemotron-reasoning", ignore_errors=True)
!git clone https://github.com/josaiahsyiem/nemotron-reasoning.git

# aggressively clear ALL vcd modules from cache
for mod in list(sys.modules):
    if mod.startswith("vcd"):
        del sys.modules[mod]

sys.path.insert(0, "/kaggle/working/nemotron-reasoning")

# re-import fresh
from vcd.solvers.registry import get_solver
from vcd.solvers.bit_manipulation import BitManipulationSolver
from vcd.solvers.text_encryption import set_vocab
from vcd.vocab import harvest_vocab
from vcd.verify.extract import extract_boxed

# reload vocab too
CSV_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
set_vocab(harvest_vocab(CSV_PATH))

# confirm solve method exists now
s = get_solver("bit_manipulation")
print("Has solve method:", hasattr(s, "solve"))

In [ ]:
test_bits_combined(df, n=100)

In [ ]:
eq = df[df["type"] == "equation_transformation"].head(6)
for _, row in eq.iterrows():
    print("EXAMPLES + TARGET:")
    print(row["prompt"].split("Below are a few examples:")[-1].strip())
    print("ANSWER:", row["answer"])
    print("=" * 60)

In [ ]:
import shutil, sys
shutil.rmtree("/kaggle/working/nemotron-reasoning", ignore_errors=True)
!git clone https://github.com/josaiahsyiem/nemotron-reasoning.git
for mod in list(sys.modules):
    if mod.startswith("vcd"):
        del sys.modules[mod]
sys.path.insert(0, "/kaggle/working/nemotron-reasoning")
from vcd.solvers.registry import get_solver
from vcd.verify.extract import extract_boxed

def build_eq_prompt(row):
    aug = get_solver("equation_transformation").augment_prompt(row["prompt"])
    messages = [{"role": "user", "content": aug + INSTRUCTION}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def test_equation(df, n=100):
    s = get_solver("equation_transformation")
    sub = df[df["type"] == "equation_transformation"].head(n)
    prompts = [build_eq_prompt(row) for _, row in sub.iterrows()]
    sampling = SamplingParams(temperature=0.3, max_tokens=1024)
    outputs = llm.generate(prompts, sampling)
    correct = sum(
        s.verify(extract_boxed(out.outputs[0].text), row["answer"])
        for (_, row), out in zip(sub.iterrows(), outputs)
    )
    print(f"equation_transformation WITH hint: {correct}/{n} = {correct/n:.0%}")

test_equation(df, n=100)

In [ ]:
print("model loaded?", "llm" in dir())
print("df loaded?", "df" in dir())

In [ ]:
import os, json
path = "/kaggle/working/traces.jsonl"
if os.path.exists(path):
    rows = [json.loads(l) for l in open(path)]
    print("Current traces.jsonl has", len(rows), "rows")
    import pandas as pd
    tdf = pd.DataFrame(rows)
    print(tdf.groupby("type")["correct"].agg(["sum", "count"]))
else:
    print("No traces.jsonl in this session")

In [ ]:
import json, os, time
from vcd.solvers.text_encryption import set_vocab
from vcd.vocab import harvest_vocab

# make sure vocab is loaded for the cipher solver
set_vocab(harvest_vocab(CSV_PATH))

OUTPUT_PATH = "/kaggle/working/traces.jsonl"

def load_done_ids(path):
    done = set()
    if os.path.exists(path):
        for line in open(path):
            try: done.add(json.loads(line)["id"])
            except: pass
    return done

def python_trace(row):
    """Try to solve fully in Python. Returns (trace, answer) or (None, None)."""
    t = row["type"]
    s = get_solver(t)
    if t == "bit_manipulation":
        ans, funcs = s.solve(row["prompt"])
        if ans is not None:
            trace, _ = s.generate_trace(row["prompt"])
            return trace, ans
    elif t == "text_encryption":
        decoded, key, steps = s.crack(row["prompt"])
        if "?" not in decoded:  # fully cracked, no ambiguous words
            trace, _ = s.generate_trace(row["prompt"])
            return trace, decoded
    return None, None

def model_prompt(row):
    """Build a hinted prompt for the model (for puzzles Python couldn't solve)."""
    s = get_solver(row["type"])
    if hasattr(s, "augment_prompt"):
        base = s.augment_prompt(row["prompt"])
    else:
        base = row["prompt"]
    messages = [{"role": "user", "content": base + INSTRUCTION}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def run_hybrid(df, types, batch_size=200):
    done = load_done_ids(OUTPUT_PATH)
    todo = df[df["type"].isin(types) & ~df["id"].isin(done)]
    print(f"To process: {len(todo)}")

    py_solved = 0
    model_queue = []  # (row, ) puzzles Python couldn't solve

    # PASS 1: Python-solvable puzzles (instant, no model)
    with open(OUTPUT_PATH, "a") as f:
        for _, row in todo.iterrows():
            trace, ans = python_trace(row)
            if ans is not None:
                s = get_solver(row["type"])
                correct = s.verify(ans, row["answer"])
                f.write(json.dumps({
                    "id": row["id"], "type": row["type"], "answer": row["answer"],
                    "predicted": ans, "correct": correct,
                    "response": trace, "source": "python",
                }) + "\n")
                if correct: py_solved += 1
            else:
                model_queue.append(row)
    print(f"Python solved: {py_solved} | Sending to model: {len(model_queue)}")

    # PASS 2: model for the rest
    sampling = SamplingParams(temperature=0.3, max_tokens=1024)
    for i in range(0, len(model_queue), batch_size):
        chunk = model_queue[i:i+batch_size]
        prompts = [model_prompt(r) for r in chunk]
        outputs = llm.generate(prompts, sampling)
        with open(OUTPUT_PATH, "a") as f:
            for row, out in zip(chunk, outputs):
                resp = out.outputs[0].text
                pred = extract_boxed(resp)
                s = get_solver(row["type"])
                correct = s.verify(pred, row["answer"])
                f.write(json.dumps({
                    "id": row["id"], "type": row["type"], "answer": row["answer"],
                    "predicted": pred, "correct": correct,
                    "response": resp, "source": "model",
                }) + "\n")
        print(f"  model batch {i//batch_size + 1}: {i+len(chunk)}/{len(model_queue)}")

    print("Done.")

In [ ]:
print("run_hybrid defined?", "run_hybrid" in dir())
print("python_trace defined?", "python_trace" in dir())
print("set_vocab worked?", "set_vocab" in dir())

In [ ]:
# Test on just 30 hard puzzles so we see it complete fast
test_sub = df[df["type"].isin(["text_encryption","bit_manipulation","equation_transformation"])].head(30)
run_hybrid(test_sub, ["text_encryption","bit_manipulation","equation_transformation"], batch_size=30)

In [ ]:
import json

# keep only the good easy-type traces; drop old hard-type test rows
keep = []
dropped = 0
for line in open("/kaggle/working/traces.jsonl"):
    row = json.loads(line)
    if row["type"] in ["numeral_conversion", "gravitational_constant", "unit_conversion"]:
        keep.append(line)
    else:
        dropped += 1

with open("/kaggle/working/traces.jsonl", "w") as f:
    f.writelines(keep)

print(f"Kept {len(keep)} easy-type traces, dropped {dropped} old hard-type rows")

In [ ]:
hard_types = ["text_encryption", "bit_manipulation", "equation_transformation"]
run_hybrid(df, hard_types, batch_size=200)

In [ ]:
import json
rows = [json.loads(l) for l in open("/kaggle/working/traces.jsonl")]
print("Total rows now:", len(rows))
import pandas as pd
tdf = pd.DataFrame(rows)
print(tdf.groupby("type")["correct"].agg(["sum", "count"]))

In [ ]:
import os
print("Size:", round(os.path.getsize("/kaggle/working/traces.jsonl")/1024/1024, 2), "MB")
print("Ready to download from Output panel")

In [ ]:
import json, random
import pandas as pd
from collections import defaultdict

TYPE_CAPS = {
    "numeral_conversion": 300, "gravitational_constant": 400,
    "unit_conversion": 700, "text_encryption": 700,
    "bit_manipulation": 607, "equation_transformation": 200,
}
INSTRUCTION = ("\n\nSolve this step by step. Put your final answer inside "
               "\\boxed{}. For example: \\boxed{your answer}")

# 1. load prompts from train.csv, keyed by id
train = pd.read_csv(CSV_PATH)
prompt_by_id = dict(zip(train["id"], train["prompt"]))

# 2. load correct traces, group by type
by_type = defaultdict(list)
with open("/kaggle/working/traces.jsonl") as f:
    for line in f:
        r = json.loads(line)
        if r.get("correct"):
            by_type[r["type"]].append(r)

# 3. sample per type, build chat examples
random.seed(42)
examples = []
for ptype, cap in TYPE_CAPS.items():
    rows = by_type[ptype]
    chosen = rows if len(rows) <= cap else random.sample(rows, cap)
    print(f"  {ptype}: {len(rows)} -> {len(chosen)}")
    for r in chosen:
        prompt = prompt_by_id.get(r["id"])
        if not prompt:
            continue
        examples.append({
            "messages": [
                {"role": "user", "content": prompt + INSTRUCTION},
                {"role": "assistant", "content": r["response"]},
            ]
        })

random.shuffle(examples)

# 4. save the training file
with open("/kaggle/working/train_data.jsonl", "w") as f:
    for ex in examples:
        f.write(json.dumps(ex) + "\n")

print(f"\nTraining set: {len(examples)} examples -> train_data.jsonl")
# show one example
print("\nSample:")
print("USER:", examples[0]["messages"][0]["content"][:150])
print("ASSISTANT:", examples[0]["messages"][1]["content"][:150])

In [ ]:
import os
p = "/kaggle/working/train_data.jsonl"
print("Exists:", os.path.exists(p))
if os.path.exists(p):
    print("Size:", round(os.path.getsize(p)/1024, 1), "KB")
    print("Lines:", sum(1 for _ in open(p)))